# xscript

> Parse and normalize YouTube caption SRT files into xscript segments

In [ ]:
#| default_exp xscript

## Design

`parse_xscript(path)` converts an SRT file into a list of normalized segments:

```python
[
  {"start": 0.08, "end": 4.88, "text": "code's not even the right verb anymore,"},
  {"start": 4.88, "end": 7.04, "text": "to my agents for 16 hours a day"},
]
```

**Pipeline:**
1. Hand-written SRT parser (no new dependency) → raw cues `(start, end, content)`
2. Per-cue normalization: splitlines → strip → drop empty → strip leading `>> ` per line → join with space
3. Rolling-window dedup: when `curr.start < prev.end`, find longest k where `prev_raw_tokens[-k:]` matches `curr_tokens[:k]` (normalized for comparison only). Drop overlap prefix; bump `start` to `prev.end`.
4. Drop cues whose text becomes empty after dedup.

**Dedup compares against prev raw tokens** (not prev deduped), so rolling-window chains A→B→C stay consistent.

In [ ]:
#| export
import re
from pathlib import Path
from yttoc.core import Segment


## Stub

In [ ]:
#| export
_TIME_RE = re.compile(r'(\d+):(\d+):(\d+),(\d+)\s*-->\s*(\d+):(\d+):(\d+),(\d+)')
_SPEAKER_RE = re.compile(r'^>>\s*')

def _ts_to_sec(h: int, m: int, s: int, ms: int) -> float:
    "Convert HH:MM:SS,mmm components to float seconds."
    return h*3600 + m*60 + s + ms/1000

def _parse_srt(content: str) -> list[tuple[float, float, str]]:
    "Parse SRT text into raw (start, end, content) tuples. Raise ValueError on malformed blocks."
    cues = []
    for block in re.split(r'\r?\n\r?\n+', content.strip()):
        lines = [l for l in block.splitlines() if l.strip()]
        if not lines: continue  # whitespace-only block
        # Find the --> line (may be at [0] if index line is missing, otherwise [1])
        time_idx = None
        for i, line in enumerate(lines):
            if _TIME_RE.search(line):
                time_idx = i
                break
        if time_idx is None:
            raise ValueError(f"Malformed SRT block (no timestamp): {block[:80]!r}")
        m = _TIME_RE.search(lines[time_idx])
        h1, m1, s1, ms1, h2, m2, s2, ms2 = map(int, m.groups())
        start = _ts_to_sec(h1, m1, s1, ms1)
        end = _ts_to_sec(h2, m2, s2, ms2)
        text_lines = lines[time_idx+1:]
        cues.append((start, end, '\n'.join(text_lines)))
    return cues

def _normalize_cue(content: str) -> str:
    "Split lines, strip each, drop empty, strip leading '>> ' per line, join with space."
    out = []
    for line in content.splitlines():
        line = line.strip()
        if not line: continue
        line = _SPEAKER_RE.sub('', line)
        if line: out.append(line)
    return ' '.join(out)

def _norm_token(t: str) -> str:
    "Normalize a token for overlap comparison (not for output)."
    return t.lower().strip('.,?!;:"\'()[]{}')

def _find_overlap(prev: list[str], curr: list[str]) -> int:
    "Return longest k where prev[-k:] matches curr[:k] under _norm_token."
    max_k = min(len(prev), len(curr))
    prev_n = [_norm_token(t) for t in prev]
    curr_n = [_norm_token(t) for t in curr]
    for k in range(max_k, 0, -1):
        if prev_n[-k:] == curr_n[:k]:
            return k
    return 0

def parse_xscript(path: str | Path # Path to SRT file
                 ) -> list[Segment]: # List of Segment objects
    "Parse SRT file into normalized xscript segments (dedup rolling-window overlap)."
    content = Path(path).read_text(encoding='utf-8')
    raw_cues = _parse_srt(content)

    segments = []
    prev_raw_tokens: list[str] = []
    prev_end = 0.0

    for start, end, raw_content in raw_cues:
        text = _normalize_cue(raw_content)
        curr_tokens = text.split()
        if not curr_tokens:
            continue

        # Dedup only when time ranges overlap
        if segments and start < prev_end:
            k = _find_overlap(prev_raw_tokens, curr_tokens)
            if k > 0:
                curr_tokens = curr_tokens[k:]
                start = prev_end
                # Preserve invariant start <= end. Can happen when the current
                # cue is fully contained within the previous cue's time window.
                if start > end:
                    end = start

        # Save raw tokens for next iteration's comparison (not deduped)
        prev_raw_tokens = text.split()
        prev_end = end

        if curr_tokens:
            segments.append(Segment(start=start, end=end, text=' '.join(curr_tokens)))

    return segments

## Tests

In [ ]:
from tempfile import NamedTemporaryFile

def _write_srt(content: str) -> Path:
    "Write SRT content to a temp file and return the path."
    f = NamedTemporaryFile(mode='w', suffix='.srt', delete=False, encoding='utf-8')
    f.write(content)
    f.close()
    return Path(f.name)

In [ ]:
# Test 1: HH:MM:SS,mmm → float seconds
srt = """1
00:00:00,080 --> 00:00:04,880
hello world
"""
segs = parse_xscript(_write_srt(srt))
assert len(segs) == 1
assert segs[0].start == 0.08
assert segs[0].end == 4.88
assert segs[0].text == 'hello world'
print('ok')

In [ ]:
# Test 2: multi-line cue → space join
srt = """1
00:00:00,000 --> 00:00:02,000
first line
second line
"""
segs = parse_xscript(_write_srt(srt))
assert len(segs) == 1
assert segs[0].text == 'first line second line'
print('ok')

In [ ]:
# Test 3: overlapping auto-caption → dedup + start bump
srt = """1
00:00:00,000 --> 00:00:05,000
code's not even the right verb anymore

2
00:00:02,000 --> 00:00:07,000
the right verb anymore to my agents
"""
segs = parse_xscript(_write_srt(srt))
assert len(segs) == 2
assert segs[0].text == "code's not even the right verb anymore"
assert segs[1].text == 'to my agents'
assert segs[1].start == 5.0  # bumped from 2.0 to prev.end
print('ok')

In [ ]:
# Test 4: non-overlapping cues → unchanged
srt = """1
00:00:00,000 --> 00:00:03,000
first sentence

2
00:00:05,000 --> 00:00:08,000
completely different
"""
segs = parse_xscript(_write_srt(srt))
assert len(segs) == 2
assert segs[0].text == 'first sentence'
assert segs[1].text == 'completely different'
assert segs[1].start == 5.0  # no bump, original
print('ok')

In [ ]:
# Test 5: full overlap → empty cue dropped
srt = """1
00:00:00,000 --> 00:00:05,000
hello world

2
00:00:02,000 --> 00:00:07,000
hello world
"""
segs = parse_xscript(_write_srt(srt))
assert len(segs) == 1
assert segs[0].text == 'hello world'
print('ok')

In [ ]:
# Test 6: start bumped past end → clamp end = start (preserve new tokens)
srt = """1
00:00:00,000 --> 00:00:10,000
hello

2
00:00:08,000 --> 00:00:09,000
hello world
"""
segs = parse_xscript(_write_srt(srt))
assert len(segs) == 2
assert segs[0].text == 'hello'
assert segs[1].text == 'world'
assert segs[1].start == 10.0  # bumped to prev.end
assert segs[1].end == 10.0  # clamped (original was 9.0)
assert segs[1].start <= segs[1].end  # invariant
print('ok')

In [ ]:
# Test 7: whitespace-only line inside cue does not split the block
srt = """1
00:00:00,000 --> 00:00:03,000
Here we go.
\u00a0
Hello everybody.

2
00:00:03,000 --> 00:00:04,000
next cue
"""
segs = parse_xscript(_write_srt(srt))
assert len(segs) == 2
assert segs[0].text == 'Here we go. Hello everybody.'
assert segs[1].text == 'next cue'
print('ok')

In [ ]:
# Test 8: malformed SRT block raises ValueError (no silent data loss)
srt = """1
00:00:00,000 --> 00:00:01,000
first

BROKEN BLOCK WITHOUT TIMING

3
00:00:03,000 --> 00:00:04,000
third
"""
try:
    parse_xscript(_write_srt(srt))
    assert False, 'should have raised ValueError'
except ValueError as e:
    assert 'BROKEN' in str(e) or 'malformed' in str(e).lower() or 'timestamp' in str(e).lower()
print('ok')

In [ ]:
#| eval: false
# Smoke test: parse a real cached SRT and show the first 15 segments.
# Requires ~/.cache/yttoc/<video_id>/captions.*.srt from yttoc-fetch.
import os
cache = Path(os.environ.get('XDG_CACHE_HOME', Path.home() / '.cache')) / 'yttoc'
for d in cache.iterdir() if cache.exists() else []:
    srt_files = sorted(d.glob('captions.*.srt'))
    if not srt_files: continue
    print(f"=== {d.name} / {srt_files[0].name} ===")
    segs = parse_xscript(srt_files[0])
    print(f"total segments: {len(segs)}")
    for s in segs[:10]:
        mm = int(s['start'] // 60)
        ss = int(s['start'] % 60)
        print(f"[{mm:02d}:{ss:02d}] {s['text']}")
    print()
    break  # just one video is enough for smoke test

## CLI

In [ ]:
#| export
import json
from fastcore.script import call_parse
from yttoc.core import fmt_duration, format_header, slice_segments, NormalizedSection, Meta
from yttoc.fetch import _DEFAULT_ROOT, _update_last_used, _glob_srt
from yttoc.toc import TocFile

def _load_segments(video_id: str, section: str, root: str | None
                  ) -> tuple[Meta, list[Segment], NormalizedSection | None, Path]:
    "Load meta, parse xscript, optionally slice to section. Return (meta, segments, sec_info, meta_path)."
    root = Path(root) if root else _DEFAULT_ROOT
    d = root / video_id
    meta_path = d / 'meta.json'
    srt_files = _glob_srt(d)
    if not (meta_path.exists() and srt_files):
        raise SystemExit(f"Not cached: {video_id}")

    meta = Meta.model_validate_json(meta_path.read_text(encoding='utf-8'))
    segments = parse_xscript(srt_files[0])
    sec_info = None

    if section:
        toc_path = d / 'toc.json'
        if not toc_path.exists():
            raise SystemExit(f"No toc.json for {video_id}. Run yttoc-toc first.")
        toc = TocFile.model_validate_json(toc_path.read_text(encoding='utf-8'))
        sec_info = next((s for s in toc.sections if s.path == section), None)
        if sec_info is None:
            raise SystemExit(f"Section {section} not found")
        segments = slice_segments(segments, sec_info.start, sec_info.end)

    return meta, segments, sec_info, meta_path

@call_parse
def yttoc_raw(video_id: str, # Exact video_id
              section: str = '', # Section path (e.g. "3"); empty for full transcript
              root: str = None, # Root cache directory (default: ~/.cache/yttoc)
             ):
    "Display transcript for a cached video (full or by section)."
    meta, segments, sec_info, meta_path = _load_segments(video_id, section, root)

    print(format_header(meta))
    print()

    if sec_info is not None:
        s_start = fmt_duration(sec_info.start)
        s_end = fmt_duration(sec_info.end)
        print(f"## {section}. {sec_info.title} ({s_start} - {s_end})")

    for s in segments:
        mm = int(s.start // 60)
        ss = int(s.start % 60)
        print(f"[{mm:02d}:{ss:02d}] {s.text}")

    _update_last_used(meta_path)

@call_parse
def yttoc_txt(video_id: str, # Exact video_id
              section: str = '', # Section path (e.g. "3"); empty for full transcript
              root: str = None, # Root cache directory (default: ~/.cache/yttoc)
             ):
    "Display transcript as plain prose with no timestamps."
    meta, segments, sec_info, meta_path = _load_segments(video_id, section, root)

    print(format_header(meta))
    print()

    if sec_info is not None:
        s_start = fmt_duration(sec_info.start)
        s_end = fmt_duration(sec_info.end)
        print(f"## {section}. {sec_info.title} ({s_start} - {s_end})")
        print()

    print(' '.join(s.text for s in segments))

    _update_last_used(meta_path)


In [ ]:
#| export
def get_xscript_range(video_id: str, # Exact video_id
                      start: int | float, # Start time in seconds
                      end: int | float, # End time in seconds
                      root: str | Path = None # Root cache directory
                     ) -> list[Segment] | dict: # List of Segment or {"error": "..."}
    "Return parsed xscript segments within [start, end). Raw parse_xscript + slice_segments output."
    root = Path(root) if root else _DEFAULT_ROOT
    d = root / video_id
    srt_files = _glob_srt(d)
    if not srt_files:
        return {'error': f'No captions found for {video_id}'}
    segments = parse_xscript(srt_files[0])
    return slice_segments(segments, start, end)

In [ ]:
# Test 9: yttoc_raw — missing video_id raises SystemExit
from tempfile import TemporaryDirectory
import io, contextlib

with TemporaryDirectory() as d:
    try:
        yttoc_raw('NONEXIST', root=d)
        assert False, 'should have raised SystemExit'
    except SystemExit:
        pass
print('ok')

In [ ]:
# Test 9: yttoc_raw — outputs header + [MM:SS] segments
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID1'; v.mkdir()
    (v / 'captions.en.srt').write_text(
        '1\n00:01:05,000 --> 00:01:08,000\nhello world\n\n'
        '2\n00:01:10,000 --> 00:01:13,000\nsecond line\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID1', 'title': 'Test Video', 'channel': 'Ch',
        'duration': 120, 'upload_date': '20260101',
        'webpage_url': 'https://youtube.com/watch?v=VID1',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2026-01-01T00:00:00+00:00',
    }))

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_raw('VID1', root=str(root))
    out = buf.getvalue()

    assert '# Test Video' in out
    assert 'Channel: Ch' in out
    assert '[01:05] hello world' in out
    assert '[01:10] second line' in out
print('ok')

In [ ]:
# Test 10: yttoc_raw — updates last_used_at
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID2'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID2', 'title': 't', 'channel': 'c',
        'duration': 1, 'upload_date': '20260101',
        'webpage_url': 'https://youtube.com/watch?v=VID2',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2000-01-01T00:00:00+00:00',
    }))

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_raw('VID2', root=str(root))

    meta = json.loads((v / 'meta.json').read_text())
    assert meta['last_used_at'] > '2000-01-01', 'last_used_at should be updated'
print('ok')

In [ ]:
# Test 11: yttoc_raw --section shows only that section's transcript
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID3'; v.mkdir()
    (v / 'captions.en.srt').write_text(
        '1\n00:00:00,000 --> 00:00:03,000\nfirst segment\n\n'
        '2\n00:00:05,000 --> 00:00:08,000\nsecond segment\n\n'
        '3\n00:00:10,000 --> 00:00:13,000\nthird segment\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID3', 'title': 'T', 'channel': 'C',
        'duration': 15, 'upload_date': '20260101',
        'webpage_url': 'https://youtube.com/watch?v=VID3',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2000-01-01T00:00:00+00:00',
    }))
    (v / 'toc.json').write_text(json.dumps({'sections': [
        {'path': '1', 'title': 'Intro', 'start': 0, 'end': 5},
        {'path': '2', 'title': 'Main', 'start': 5, 'end': 15},
    ]}))

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_raw('VID3', section='2', root=str(root))
    out = buf.getvalue()

    assert '## 2. Main (0:05 - 0:15)' in out
    assert '[00:05] second segment' in out
    assert '[00:10] third segment' in out
    assert 'first segment' not in out  # section 1 excluded
print('ok')

In [ ]:
# Test 12: yttoc_txt — outputs header + plain text joined, no timestamps
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID_TXT'; v.mkdir()
    (v / 'captions.en.srt').write_text(
        '1\n00:00:00,000 --> 00:00:03,000\nfirst segment\n\n'
        '2\n00:00:05,000 --> 00:00:08,000\nsecond segment\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID_TXT', 'title': 'Test', 'channel': 'C',
        'duration': 10, 'upload_date': '20260101',
        'webpage_url': 'https://youtube.com/watch?v=VID_TXT',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2000-01-01T00:00:00+00:00',
    }))

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_txt('VID_TXT', root=str(root))
    out = buf.getvalue()

    assert '# Test' in out
    assert 'first segment second segment' in out
    assert '[00:' not in out  # no timestamps
print('ok')

In [ ]:
# Test 13: yttoc_txt --section shows only that section's prose
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID_TXT2'; v.mkdir()
    (v / 'captions.en.srt').write_text(
        '1\n00:00:00,000 --> 00:00:03,000\nfirst segment\n\n'
        '2\n00:00:05,000 --> 00:00:08,000\nsecond segment\n\n'
        '3\n00:00:10,000 --> 00:00:13,000\nthird segment\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID_TXT2', 'title': 'T', 'channel': 'C',
        'duration': 15, 'upload_date': '20260101',
        'webpage_url': 'https://youtube.com/watch?v=VID_TXT2',
        'description': '', 'captions': {'en': 'auto'},
        'last_used_at': '2000-01-01T00:00:00+00:00',
    }))
    (v / 'toc.json').write_text(json.dumps({'sections': [
        {'path': '1', 'title': 'Intro', 'start': 0, 'end': 5},
        {'path': '2', 'title': 'Main', 'start': 5, 'end': 15},
    ]}))

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_txt('VID_TXT2', section='2', root=str(root))
    out = buf.getvalue()

    assert '## 2. Main (0:05 - 0:15)' in out
    assert 'second segment third segment' in out
    assert 'first segment' not in out  # section 1 excluded
    assert '[00:' not in out  # no timestamps
print('ok')

In [ ]:
# Test 14: get_xscript_range returns sliced segments matching parse_xscript output shape
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID_GXR'; v.mkdir()
    (v / 'captions.en.srt').write_text(
        '1\n00:00:00,000 --> 00:00:03,000\nfirst\n\n'
        '2\n00:00:05,000 --> 00:00:08,000\nsecond\n\n'
        '3\n00:00:10,000 --> 00:00:13,000\nthird\n')

    result = get_xscript_range('VID_GXR', 5, 15, root)
    assert isinstance(result, list)
    assert len(result) == 2
    assert result[0].text == 'second'
    assert result[1].text == 'third'
    assert isinstance(result[0].start, float)
    assert isinstance(result[0].end, float)
    assert hasattr(result[0], 'start') and hasattr(result[0], 'end') and hasattr(result[0], 'text')
print('ok')

In [ ]:
# Test 15: get_xscript_range returns error dict when SRT missing
with TemporaryDirectory() as d:
    result = get_xscript_range('NONEXIST', 0, 100, Path(d))
    assert 'error' in result
print('ok')

In [ ]:
# Test 16: get_xscript_range with no matching segments returns empty list
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID_EMPTY'; v.mkdir()
    (v / 'captions.en.srt').write_text(
        '1\n00:00:00,000 --> 00:00:03,000\nhello\n')

    result = get_xscript_range('VID_EMPTY', 100, 200, root)
    assert result == []
print('ok')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()